# exp_e2_mpnet_multi — Semantic Graph Builder v2

**Phase 1a, embedder = `paraphrase-multilingual-mpnet-base-v2`, LLM = `deepseek-v32/latest` (API).**

Запускается из `exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/` — все пути разрешаются автоматически от `clustering_1/`.

Перед запуском: `export YANDEX_CLOUD_API_KEY=...`.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

In [2]:
import os, sys, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# layout: clustering_1/exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/
EXP_DIR   = Path().resolve()
REPO_ROOT = EXP_DIR.parents[3]                 # clustering_1/
LLM_V2    = REPO_ROOT / 'llm_v2'

# put repo root on sys.path so `import llm_v2` works
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
# guard: never expose llm_v2/ as flat path
if str(LLM_V2) in sys.path:
    sys.path.remove(str(LLM_V2))

from llm_v2.config_schema import load_config
config = load_config(EXP_DIR / 'config.yaml')

# expand ${YANDEX_CLOUD_API_KEY} etc. (no-op for local LLMs)
config.llm.api_key = os.path.expandvars(config.llm.api_key)
config.llm.base_url = os.path.expandvars(config.llm.base_url)
config.llm.folder = os.path.expandvars(config.llm.folder)

print('EXP_DIR  :', EXP_DIR)
print('REPO_ROOT:', REPO_ROOT)
print('LLM_V2   :', LLM_V2)
print()
print(config.model_dump_json(indent=2))

EXP_DIR  : /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e4_bge_m3
REPO_ROOT: /home/platoon/graph/semantic-graph
LLM_V2   : /home/platoon/graph/semantic-graph/llm_v2

{
  "llm": {
    "provider": "api",
    "model_name": "deepseek-v32/latest",
    "max_new_tokens": 500,
    "temperature": 0.3,
    "device": "cpu",
    "load_in_8bit": false,
    "api_key": "${YANDEX_CLOUD_API_KEY}",
    "base_url": "https://ai.api.cloud.yandex.net/v1",
    "folder": "b1gpiug3vgbpe1cb4e5c",
    "instructions": ""
  },
  "embedding": {
    "model_name": "BAAI/bge-m3",
    "device": "cuda"
  },
  "coreference": {
    "enabled": false,
    "prompt_file": "prompts/coreference_ru.txt",
    "context_sentences": 3,
    "window_sentences": 5
  },
  "extraction": {
    "prompt_file": "prompts/extraction_ru.txt",
    "chunk_size": 3,
    "overlap_size": 1
  },
  "normalization": {
    "enabled": true,
    "language": "ru"
  },
  "deduplication": {
    "enabled": true,
    "threshold": 0.

In [ ]:
config.llm.api_key = ""

In [4]:
from llm_v2.models.llm_client import LLMClient
from llm_v2.models.embedder import Embedder

llm = LLMClient(config.llm)
embedder = Embedder(config.embedding)
print(f'LLM loaded: {config.llm.model_name}')
print(f'Embedder loaded: {config.embedding.model_name} (dim={embedder.dim})')

/home/platoon/graph/graph_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-07 15:18:59,492 [INFO] Load pretrained SentenceTransformer: BAAI/bge-m3
2026-05-07 15:19:00,009 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 15:19:00,185 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
2026-05-07 15:19:00,354 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 15:19:00,525 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sente

LLM loaded: deepseek-v32/latest
Embedder loaded: BAAI/bge-m3 (dim=1024)


In [5]:
from llm_v2.utils.io import load_text

input_path = Path(config.paths.input_text)
if not input_path.is_absolute():
    input_path = (LLM_V2 / input_path).resolve()
text = load_text(input_path)
print(f'Input: {input_path}')
print(f'Length: {len(text)} chars')
print(text[:500])

Input: /home/platoon/graph/semantic-graph/benchmark/final_bench/formated_fragment2.md
Length: 15466 chars
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.

Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.

В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам 


## [0] Preprocessing

In [6]:
from llm_v2.stages.preprocessing import preprocess

sentences = preprocess(text, language=config.normalization.language)
for s in sentences:
    print(f'  [{s.id}] {s.text}')

  [0] # Линейная классификация

Теперь давайте поговорим про задачу классификации.
  [1] Для начала будем говорить про бинарную классификацию на два класса.
  [2] Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.
  [3] Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.
  [4] В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$.
  [5] Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого.
  [6] **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой.
  [7] Выборка, для которой это возможно, называется линейно разделимой.
  [8] Увы, в реальной жизни такое встречается кр

## [1] Coreference Resolution

In [7]:
from llm_v2.stages.coreference import resolve_coreferences

resolved_text, sentences = resolve_coreferences(
    sentences, llm, config.coreference, base_dir=LLM_V2
)
print('Resolved text:')
print(resolved_text)
print(f'\nSentences after coref: {len(sentences)}')

Resolved text:
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда. Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$. В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$. Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого. **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой. Выборка, для которой это возможно, называется линейно разделимой. Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модель

## [1.5] Chunking

In [8]:
from llm_v2.stages.chunking import build_chunks

chunks = build_chunks(sentences, config.extraction)
for c in chunks:
    print(f'  {c.id} (sents {c.sentence_ids}): {c.text[:80]}...')

  chunk_0 (sents [0, 1, 2]): # Линейная классификация

Теперь давайте поговорим про задачу классификации. Для...
  chunk_1 (sents [2, 3, 4]): Обобщить эту задачу до задачи классификации на $K$ классов не составит большого ...
  chunk_2 (sents [4, 5, 6]): В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам буду...
  chunk_3 (sents [6, 7, 8]): **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: пол...
  chunk_4 (sents [8, 9, 10]): Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модел...
  chunk_5 (sents [10, 11, 12]): $$

<details>
<summary>Почему бы не решать задачу классификации как задачу регре...
  chunk_6 (sents [12, 13, 14]): Во вторых, ошибкой будет считаться предсказание, например, $5$ вместо $1$, хотя ...
  chunk_7 (sents [14, 15, 16]): </details>

Сконструируем теперь функционал ошибки так, чтобы он вышеперечисленн...
  chunk_8 (sents [16, 17, 18]): $$

Домножим обе части на $y_i$ и немного упростим:

$

## [2] Triplet Extraction

In [9]:
from llm_v2.stages.extraction import extract_triplets

raw_triplets = extract_triplets(chunks, llm, config.extraction, base_dir=LLM_V2)
print(f'Extracted {len(raw_triplets)} raw triplets:')
for t in raw_triplets:
    print(f'  {t.subject} | {t.relation} | {t.object}  [{t.chunk_id}]')

Extracting triplets: 100%|██████████| 56/56 [10:09<00:00, 10.89s/it]

Extracted 344 raw triplets:
  линейная классификация | является темой | классификация  [chunk_0]
  мы | будем говорить про | задача классификации  [chunk_0]
  задача классификации | является | бинарная классификация  [chunk_0]
  бинарная классификация | классифицирует на | два класса  [chunk_0]
  задача классификации | обобщается до | классификация на K классов  [chunk_0]
  классификация на K классов | не составляет труда | обобщение  [chunk_0]
  задача | обобщается до | задача классификации  [chunk_1]
  задача классификации | имеет количество классов | K классов  [chunk_1]
  таргеты y | кодируют принадлежность | класс  [chunk_1]
  класс | может быть | положительный класс  [chunk_1]
  класс | может быть | отрицательный класс  [chunk_1]
  таргеты y | принадлежат множеству | -1  [chunk_1]
  таргеты y | принадлежат множеству | 1  [chunk_1]
  x | является | векторы  [chunk_1]
  векторы | принадлежат пространству | R^D  [chunk_1]
  метки классов | могут быть | 0  [chunk_1]
  метки классов |

## [3] Normalization

In [10]:
from llm_v2.stages.normalization import normalize_triplets

norm_triplets = normalize_triplets(raw_triplets, config.normalization)
print(f'Normalized {len(norm_triplets)} triplets:')
for t in norm_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

2026-05-07 15:29:36,552 [INFO] Loading dictionaries from /home/platoon/graph/graph_env/lib/python3.12/site-packages/pymorphy3_dicts_ru/data
2026-05-07 15:29:36,580 [INFO] format: 2.4, revision: 417150, updated: 2022-01-08T22:09:24.565962


Normalized 344 triplets:
  линейный классификация | являться тема | классификация
  мы | быть говорить про | задача классификация
  задача классификация | являться | бинарный классификация
  бинарный классификация | классифицировать на | два класс
  задача классификация | обобщаться до | классификация на k класс
  классификация на k класс | не составлять труд | обобщение
  задача | обобщаться до | задача классификация
  задача классификация | иметь количество класс | k класс
  таргет y | кодировать принадлежность | класс
  класс | мочь быть | положительный класс
  класс | мочь быть | отрицательный класс
  таргет y | принадлежать множество | -1
  таргет y | принадлежать множество | 1
  x | являться | вектор
  вектор | принадлежать пространство | R^D
  метка класс | мочь быть | 0
  метка класс | мочь быть | 1
  мы | договориться обозначать | класс
  мы | быть встречать | метка {0,1}
  мы | хотеть обучить | линейный модель
  линейный модель | задавать | плоскость
  плоскость | должный отд

## [4] Deduplication

In [11]:
from llm_v2.stages.deduplication import deduplicate_triplets

dedup_triplets = deduplicate_triplets(norm_triplets, embedder, config.deduplication)
print(f'After dedup: {len(norm_triplets)} -> {len(dedup_triplets)} triplets')
for t in dedup_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

After dedup: 344 -> 317 triplets
  линейный классификация | являться тема | классификация
  мы | быть говорить про | задача классификация
  задача классификация | являться | бинарный классификация
  бинарный классификация | классифицировать на | два класс
  задача классификация | обобщаться до | классификация на k класс
  классификация на k класс | не составлять труд | обобщение
  задача | обобщаться до | задача классификация
  задача классификация | иметь количество класс | k класс
  таргет y | кодировать принадлежность | класс
  класс | мочь быть | положительный класс
  класс | мочь быть | отрицательный класс
  таргет y | принадлежать множество | -1
  таргет y | принадлежать множество | 1
  x | являться | вектор
  вектор | принадлежать пространство | R^D
  метка класс | мочь быть | 0
  метка класс | мочь быть | 1
  мы | договориться обозначать | класс
  мы | быть встречать | метка {0,1}
  мы | хотеть обучить | линейный модель
  линейный модель | задавать | плоскость
  плоскость | дол

## [5] Graph Assembly (raw)

In [12]:
from llm_v2.stages.graph_assembly import assemble_graph

raw_graph = assemble_graph(dedup_triplets, chunks, text, config)
print(f'Raw graph: {len(raw_graph.nodes)} nodes, {len(raw_graph.edges)} edges')
print('\nNodes:')
for n in raw_graph.nodes:
    print(f'  {n.id}: {n.label} ({len(n.mentions)} mentions)')
print('\nEdges:')
for e in raw_graph.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (w={e.weight})')

Raw graph: 343 nodes, 317 edges

Nodes:
  n0: линейный классификация (1 mentions)
  n1: классификация (4 mentions)
  n2: мы (35 mentions)
  n3: задача классификация (6 mentions)
  n4: бинарный классификация (2 mentions)
  n5: два класс (1 mentions)
  n6: классификация на k класс (2 mentions)
  n7: обобщение (1 mentions)
  n8: задача (3 mentions)
  n9: k класс (1 mentions)
  n10: таргет y (6 mentions)
  n11: класс (13 mentions)
  n12: положительный класс (3 mentions)
  n13: отрицательный класс (3 mentions)
  n14: -1 (2 mentions)
  n15: 1 (2 mentions)
  n16: x (1 mentions)
  n17: вектор (2 mentions)
  n18: R^D (1 mentions)
  n19: метка класс (4 mentions)
  n20: 0 (2 mentions)
  n21: метка {0,1} (1 mentions)
  n22: линейный модель (6 mentions)
  n23: плоскость (6 mentions)
  n24: объект один класс (1 mentions)
  n25: выборка (1 mentions)
  n26: линейно разделимый (1 mentions)
  n27: линейно разделимый выборка (1 mentions)
  n28: идеальный ситуация (1 mentions)
  n29:  (2 mentions)
  n30: 

## [6] Clustering

In [13]:
from llm_v2.stages.clustering import cluster_graph, cluster_graph_multi, cluster_graph_all_methods
from llm_v2.utils.io import load_prompt

naming_prompt_path = Path(config.clustering.cluster_naming_prompt)
if not naming_prompt_path.is_absolute():
    naming_prompt_path = (LLM_V2 / naming_prompt_path).resolve()
naming_prompt = load_prompt(naming_prompt_path) if naming_prompt_path.exists() else None

if config.clustering.multi_method:
    multi = cluster_graph_all_methods(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    print('Multi-method clustering:')
    for method_name, mr in multi.methods.items():
        print(f'  {method_name}: {len(mr.param_labels)} variants')
        for lbl in mr.param_labels:
            g = mr.graphs[lbl]
            print(f'    {lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    agg = multi.methods['agglomerative']
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
elif config.clustering.is_multi_threshold:
    multi = cluster_graph_multi(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    agg = multi.methods['agglomerative']
    print(f'Multi-threshold: {len(agg.param_labels)} levels')
    for lbl in agg.param_labels:
        g = agg.graphs[lbl]
        print(f'  t={lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
else:
    clustered = cluster_graph(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )

print(f'\nClustered graph: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print('\nClustered Nodes:')
for n in clustered.nodes:
    print(f'  {n.id}: {n.label} (members={n.members}, size={n.size})')
print('\nClustered Edges:')
for e in clustered.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (size={e.size})')

Multi-method clustering:
  agglomerative: 10 variants
    0.250: 211 nodes, 228 edges
    0.322: 138 nodes, 183 edges
    0.394: 97 nodes, 150 edges
    0.467: 73 nodes, 121 edges
    0.539: 53 nodes, 99 edges
    0.611: 45 nodes, 90 edges
    0.683: 39 nodes, 88 edges
    0.756: 31 nodes, 83 edges
    0.828: 19 nodes, 61 edges
    0.900: 9 nodes, 20 edges
  kmeans: 4 variants
    k=10: 10 nodes, 41 edges
    k=25: 25 nodes, 71 edges
    k=40: 40 nodes, 92 edges
    k=55: 55 nodes, 101 edges
  hdbscan: 9 variants
    mcs=3,ms=1: 69 nodes, 115 edges
    mcs=3,ms=3: 76 nodes, 124 edges
    mcs=3,ms=5: 97 nodes, 137 edges
    mcs=5,ms=1: 64 nodes, 103 edges
    mcs=5,ms=3: 76 nodes, 119 edges
    mcs=5,ms=5: 97 nodes, 137 edges
    mcs=10,ms=1: 111 nodes, 133 edges
    mcs=10,ms=3: 128 nodes, 145 edges
    mcs=10,ms=5: 167 nodes, 177 edges

Clustered graph: 45 nodes, 90 edges

Clustered Nodes:
  c0:  (members=['n21', 'n23', 'n24', 'n25', 'n26', 'n27', 'n28', 'n29', 'n30', 'n31'], size=10)

## Save outputs

In [14]:
from llm_v2.utils.io import save_json, save_text

out = EXP_DIR / config.paths.output_dir
out.mkdir(parents=True, exist_ok=True)

save_text(resolved_text, out / 'coreference_resolved.txt')
save_json(raw_graph.model_dump(), out / 'raw_graph.json')
save_json(clustered.model_dump(), out / 'clustered_graph.json')

if config.clustering.multi_method or config.clustering.is_multi_threshold:
    save_json(multi.model_dump(), out / 'multi_clustered_graph.json')
    method_counts = {m: len(r.param_labels) for m, r in multi.methods.items()}
    print(f'Saved multi_clustered_graph.json (methods: {method_counts})')

print(f'Saved to {out}/')

Saved multi_clustered_graph.json (methods: {'agglomerative': 10, 'kmeans': 4, 'hdbscan': 9})
Saved to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e4_bge_m3/output/


## Benchmark vs ground-truth graph

In [15]:
from llm_v2.benchmark import (
    evaluate_graph,
    evaluate_multi_graph,
    load_clustered_graph,
    print_metrics,
    print_multi_metrics,
    best_variant,
    show_node_alignments,
    show_edge_alignments,
    multi_metrics_to_dict,
)

# GT inputs (absolute, robust to CWD)
gt_graph_path = REPO_ROOT / 'benchmark' / 'final_bench' / 'graph_clustered.json'
gt_text_path  = REPO_ROOT / 'benchmark' / 'final_bench' / 'formated_fragment2.md'

gt_graph = load_clustered_graph(gt_graph_path)
gt_text  = gt_text_path.read_text(encoding='utf-8')

# embedding context: prefer the coreference-resolved text the pipeline saw
source_text = resolved_text if resolved_text else gt_text

print(f'GT  : {len(gt_graph.nodes)} nodes, {len(gt_graph.edges)} edges  ({gt_graph_path})')
print(f'Pred: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print(f'Context text: {len(source_text)} chars')

TAU_NODE = 0.6
TAU_EDGE = 0.6
BETA = 1.0
NODE_WEIGHT = 0.6
EDGE_WEIGHT = 0.4
NODE_WINDOW = 300
EDGE_WINDOW = 400
TOP_K = 10

GT  : 55 nodes, 51 edges  (/home/platoon/graph/semantic-graph/benchmark/final_bench/graph_clustered.json)
Pred: 45 nodes, 90 edges
Context text: 15418 chars


In [16]:
metrics = evaluate_graph(
    pred=clustered,
    gt=gt_graph,
    source_text=source_text,
    embedder=embedder,
    tau_node=TAU_NODE,
    tau_edge=TAU_EDGE,
    beta=BETA,
    node_weight=NODE_WEIGHT,
    edge_weight=EDGE_WEIGHT,
    node_window=NODE_WINDOW,
    edge_window=EDGE_WINDOW,
)

print_metrics(metrics)
metrics.summary()

Batches: 100%|██████████| 1/1 [00:00<00:00, 123.11it/s]


GraphScore = 0.6·NodeF1 + 0.4·EdgeF1  =  0.2431

Nodes (pred=45, gt=55, matched=32, tau=0.6, beta=1):
  TP(soft)  = 12.4281
  precision = 0.2762
  recall    = 0.2260
  F1        = 0.2486
Edges (pred=90, gt=51, matched=42, tau=0.6, beta=1):
  TP(soft)  = 16.5691
  precision = 0.1841
  recall    = 0.3249
  F1        = 0.2350


{'graph_score': 0.24314658757980834,
 'node_weight': 0.6,
 'edge_weight': 0.4,
 'nodes': {'precision': 0.27618006732728745,
  'recall': 0.225965509631417,
  'f_beta': 0.24856206059455874,
  'beta': 1.0,
  'tau': 0.6,
  'tp': 12.428103029727936,
  'pred_count': 45,
  'gt_count': 55,
  'matched_count': 32},
 'edges': {'precision': 0.18410164614518484,
  'recall': 0.32488525790326733,
  'f_beta': 0.23502337805768278,
  'beta': 1.0,
  'tau': 0.6,
  'tp': 16.569148153066635,
  'pred_count': 90,
  'gt_count': 51,
  'matched_count': 42},
 'pred_structure': {'n_nodes': 45,
  'n_edges': 90,
  'density': 0.045454545454545456,
  'n_components': 1,
  'n_isolated': 0,
  'component_sizes': [45],
  'component_size_min': 45,
  'component_size_max': 45,
  'component_size_mean': 45.0,
  'component_size_quantiles': {'q25': 45.0,
   'q50': 45.0,
   'q75': 45.0,
   'q90': 45.0}},
 'gt_structure': {'n_nodes': 55,
  'n_edges': 51,
  'density': 0.01717171717171717,
  'n_components': 6,
  'n_isolated': 0,
  'c

In [17]:
show_node_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

Matched node pairs: 32 / min(45, 55)=45

Top 10 matched (by quality q):
  [q=0.922]  'p = 1/2'  ↔  'гиперплоскость'
  [q=0.867]  'множество точка класс 0'  ↔  'разделяющая поверхность'
  [q=0.838]  'регрессия'  ↔  'MSE'
  [q=0.829]  'задача'  ↔  'стохастический градиентный спуск'
  [q=0.802]  'тут'  ↔  'логистическая регрессия'
  [q=0.802]  'метод опорный вектор'  ↔  'вектор'
  [q=0.794]  'поиск'  ↔  'регрессия'
  [q=0.779]  ''  ↔  'выборка'
  [q=0.613]  'картинка'  ↔  'число ошибок классификатора'
  [q=0.561]  'ноль'  ↔  'минимальный отступ'

Bottom 10 matched:
  [q=0.132]  'отступ'  ↔  'положительный отступ'
  [q=0.132]  'отсечка'  ↔  'разделяющая плоскость'
  [q=0.116]  'вопрос'  ↔  '$\\nabla_w L(w,x,y)=2\\lambda w+\\sum_i\\begin{cases}0, & 1-y_i\\langle w,x_i\\rangle\\le 0, \\\\ -y_i x_i, & 1-y_i\\langle w,x_i\\rangle>0.\\end{cases}$'
  [q=0.114]  'два класс'  ↔  'задача классификации'
  [q=0.102]  'особенность'  ↔  'функция потерь $L(w,X,y)$'
  [q=0.079]  'по-другому'  ↔  'линейна

In [18]:
show_edge_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

Matched edge pairs: 42 / min(90, 51)=51

Top 10 matched (by quality q):
  [q=0.901]  регрессия —[иметь название]→ правда
           ↔  регрессия —[может быть наивным подходом к]→ задача классификации
  [q=0.859]  картинка —[важный]→ регрессия
           ↔  регрессия —[является плохим подходом для]→ задача классификации
  [q=0.788]  правда —[предсказывать]→ доля положительный и отрицательный класс
           ↔  логистическая регрессия —[задаёт]→ разделяющая поверхность
  [q=0.782]  ∇_w l(y,x,w)= -∑_i x_i(y_i-σ(⟨w,x_i⟩)) —[иметь]→ $p$
           ↔  правдоподобие $p(y\mid X,w)$ —[основано на]→ распределение Бернулли
  [q=0.771]  правда —[выдавать]→ 1
           ↔  разделяющая поверхность —[является]→ гиперплоскость
  [q=0.757]  тут —[являться задача]→ поиск
           ↔  логистическая регрессия —[предсказывает]→ вероятность $p$
  [q=0.725]  1 —[обозначаться]→ тут
           ↔  минимальный отступ —[соответствует]→ ширина полосы $\dfrac{2}{\|w\|_2}$
  [q=0.692]  регрессия —[лежать не с тот 

In [19]:
save_json(metrics.summary(), out / 'benchmark_metrics.json')
print(f'Saved benchmark_metrics.json to {out}/')

Saved benchmark_metrics.json to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e4_bge_m3/output/


## Benchmark — multi-method / multi-threshold sweep

In [20]:
is_multi = config.clustering.multi_method or config.clustering.is_multi_threshold

if not is_multi:
    print('Skipped: multi-method / multi-threshold not enabled in config')
    multi_metrics = None
else:
    total = sum(len(mr.graphs) for mr in multi.methods.values())
    print(f'Evaluating {total} configurations...')
    multi_metrics = evaluate_multi_graph(
        multi=multi,
        gt=gt_graph,
        source_text=source_text,
        embedder=embedder,
        tau_node=TAU_NODE,
        tau_edge=TAU_EDGE,
        beta=BETA,
        node_weight=NODE_WEIGHT,
        edge_weight=EDGE_WEIGHT,
        node_window=NODE_WINDOW,
        edge_window=EDGE_WINDOW,
    )
    print(f'Done: {sum(len(v) for v in multi_metrics.values())} variants evaluated')

Evaluating 23 configurations...


Batches: 100%|██████████| 1/1 [00:00<00:00, 118.33it/s]


Done: 23 variants evaluated


In [21]:
if multi_metrics:
    print_multi_metrics(multi_metrics, sort_by='graph_score')

method        param                  pred_n pred_e  matched_n  matched_e    P_n    R_n    F_n    P_e    R_e    F_e   graph
--------------------------------------------------------------------------------------------------------------------------
agglomerative 0.539                      53     99         36         46  0.264  0.254  0.259  0.176  0.341  0.232  0.2482
agglomerative 0.683                      39     88         28         43  0.303  0.215  0.252  0.188  0.324  0.238  0.2461
agglomerative 0.756                      31     83         26         41  0.347  0.196  0.250  0.191  0.311  0.237  0.2450
agglomerative 0.611                      45     90         32         42  0.276  0.226  0.249  0.184  0.325  0.235  0.2431
agglomerative 0.467                      73    121         40         45  0.212  0.282  0.242  0.164  0.389  0.231  0.2374
agglomerative 0.828                      19     61         17         33  0.464  0.160  0.238  0.181  0.217  0.197  0.2219
agglomerative 0.

In [22]:
if multi_metrics:
    method, param, best_m = best_variant(multi_metrics, by='graph_score')
    best_graph = multi.methods[method].graphs[param]
    print(f'Best variant: method={method}, param={param}')
    print(f'  graph: {len(best_graph.nodes)} nodes, {len(best_graph.edges)} edges')
    print()
    print_metrics(best_m)

Best variant: method=kmeans, param=k=25
  graph: 25 nodes, 71 edges

GraphScore = 0.6·NodeF1 + 0.4·EdgeF1  =  0.2581

Nodes (pred=25, gt=55, matched=24, tau=0.6, beta=1):
  TP(soft)  = 10.2510
  precision = 0.4100
  recall    = 0.1864
  F1        = 0.2563
Edges (pred=71, gt=51, matched=40, tau=0.6, beta=1):
  TP(soft)  = 15.9159
  precision = 0.2242
  recall    = 0.3121
  F1        = 0.2609


In [23]:
if multi_metrics:
    save_json(multi_metrics_to_dict(multi_metrics), out / 'benchmark_metrics_multi.json')
    print(f'Saved benchmark_metrics_multi.json to {out}/')

Saved benchmark_metrics_multi.json to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e4_bge_m3/output/
